# Edge space

`CompositeEdges` is the space of an electric field next to the potential's nodes: one value per
edge of the leaf cells, the line integral of the field along it. It is the field space of an
implicit, energy-conserving particle-in-cell scheme on patches: the gradient maps the nodes'
potential into it exactly, its mass matrix is the field energy, and the gather of the field at
markers has the current deposit as its transpose.

In [ ]:
import numpy as np

from patchwork import (
    AMRHierarchy,
    CompositeEdges,
    CompositeNodes,
    Dirichlet,
    Neumann,
    NodalPoisson,
    plot_patches,
)

## Edges on a refined mesh

Inside a leaf cell the field is the lowest-order edge interpolant: the x component varies
linearly in y between the cell's lower and upper x-edges, the y component linearly in x between
its left and right y-edges. A coarse edge on the edge of a patch is two fine half-edges; they
hang, and each carries half the coarse edge's line integral (`edges.reduction`).

In [ ]:
mesh = AMRHierarchy(base_shape=(16, 16), periodic=(False, True))
mesh.add_patch(level=1, box=((8, 24), (8, 24)))
nodes = CompositeNodes(mesh)
edges = CompositeEdges(nodes)
print(
    f"{nodes.num_nodes} nodes ({nodes.hanging.sum()} hanging), "
    f"{edges.num_edges} edges ({edges.hanging.sum()} hanging)"
)
fig, ax = plot_patches(mesh)

## The gradient

`edges.gradient` is the difference of the potential along every edge. For a potential with its
hanging nodes filled (`nodes.reduction @ phi`) the result is an edge field whose hanging edges
are filled too: the constraints commute with the gradient. The field it gathers at any point is
the gradient of the bilinear interpolant of the potential.

In [ ]:
x, y = nodes.coordinates.T
phi = nodes.reduction @ (np.sin(2 * np.pi * x) * np.cos(2 * np.pi * y))
e = edges.gradient @ phi
print("hanging edges filled:", np.allclose(edges.reduction @ e, e))

points = np.array([[0.3, 0.52], [0.7, 0.45]])
ex, ey = (matrix @ e for matrix in edges.evaluation(points))
step = 1e-7
dx = (
    nodes.evaluation(points + [[step], [0]]) @ phi
    - nodes.evaluation(points - [[step], [0]]) @ phi
) / (2 * step)
print("E_x from the edges:", ex, " d(phi)/dx of the bilinear potential:", dx)

## The mass matrix and the energy

`edges.mass(lumped=True)` integrates `|E|^2` with the vertex quadrature: it is diagonal, and
`G^T M1 G` is `NodalPoisson`'s operator, its five-point stencil inside a patch.
`mass(lumped=False)` is the exact integral; both give the same energy for a uniform field.

In [ ]:
poisson = NodalPoisson(
    mesh, faces={"x_lower": Dirichlet(0.0), "x_upper": Neumann(0.0)}, nodes=nodes
)
gradient = edges.gradient @ nodes.reduction
stiffness = gradient.T @ edges.mass(lumped=True) @ gradient
free = poisson.free
print(
    "difference to NodalPoisson:",
    abs(stiffness.tocsr()[free][:, free] - poisson.matrix).max(),
)
print(
    "field energy, lumped and exact:",
    e @ edges.mass() @ e,
    e @ edges.mass(lumped=False) @ e,
)

## Gather and deposit

`edges.deposit(points, vectors)` puts vectors at points on the edges with the weights of the
gather, whose transpose it is: the work of the gathered field on the markers' currents is the
current's work on the edges.

In [ ]:
rng = np.random.default_rng(1)
points = np.vstack((rng.random(500), rng.random(500)))
current = rng.normal(size=(2, 500))
gathered = np.array([matrix @ e for matrix in edges.evaluation(points)])
print(edges.deposit(points, current) @ e, np.sum(current * gathered))

## The current along paths

`edges.deposit_paths(start, end, charges)` integrates the current of charges moving along
straight paths against the edge functions, each path split at every cell face it crosses,
patch edges included. The charge is then conserved exactly: the divergence of the current,
`(G T)^T j`, is the change of the charge deposited on the nodes.

In [ ]:
start = np.vstack((rng.random(500), rng.random(500)))
end = start + rng.normal(scale=0.1, size=start.shape)
end[0] = np.clip(end[0], 0.0, 1.0)  # bounded in x, periodic in y
charges = rng.normal(size=500)
current = edges.deposit_paths(start, end, charges)
divergence = gradient.T @ current
change = nodes.reduction.T @ (
    nodes.deposit(end, charges) - nodes.deposit(start, charges)
)
print("largest difference:", np.abs(divergence - change).max())